# Time Series Benchmarking Framework

Interactive notebook for running forecast horse-races.  
**Workflow:** Setup series → Configure knobs → Run benchmark → Inspect results

In [1]:
import os
import importlib.util
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from benchmark import (
    TimeSeries, SeriesRegistry,
    MeanForecaster, ARIMAForecaster, BayesianARForecaster, SSAForecaster, TimesFMForecaster,
    BenchmarkRunner, BenchmarkResults,
    ReplicatedBenchmarkRunner, ReplicatedBenchmarkResults,
)
spec = importlib.util.spec_from_file_location("local_secrets", "./secrets.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
HF_token = mod.HF_token
os.environ["HF_TOKEN"] = HF_token

## 1. Setup Phase — Register Series

Load the default synthetic DGPs and (optionally) register your own.

In [2]:
SeriesRegistry.clear()
SeriesRegistry.register_defaults(n=100, seed=42)

print("Available series:", SeriesRegistry.list_available())

Available series: ['AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)']


In [3]:
# --- Register your own series (uncomment / adapt) ---

# From a CSV:# SeriesRegistry.register_from_csv("SP500", path="data/sp500.csv", column="Close")

# From a raw array:
# my_values = np.random.randn(500).cumsum()
# SeriesRegistry.register("Random Walk", TimeSeries.from_array(my_values, name="Random Walk"))

print("Available series:", SeriesRegistry.list_available())

Available series: ['AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)']


## 2. Monte Carlo Replications — Same DGP, Multiple Seeds

Run the *same* DGP with many different random seeds so each replication
draws new AR/MA coefficients **and** new innovations.  This lets you
compare forecasters on the *distribution* of performance rather than a
single lucky/unlucky draw.

In [4]:
# ── DGP and replication settings ────────────────────────────────

mc_n_points = widgets.BoundedIntText(
    value=100, min=20, max=50_000,
    description='# points (n):',
    style={'description_width': 'initial'},
)

dgp_options = {
    "AR(2)":        lambda seed: TimeSeries.from_ar(p=2, n=mc_n_points.value, seed=seed),
    "AR(5)":        lambda seed: TimeSeries.from_ar(p=5, n=mc_n_points.value, seed=seed),
    "ARMA(2,2)":    lambda seed: TimeSeries.from_arma(p=2, q=2, n=mc_n_points.value, seed=seed),
    "ARMA(5,5)":    lambda seed: TimeSeries.from_arma(p=5, q=5, n=mc_n_points.value, seed=seed),
    "Seasonal(12)": lambda seed: TimeSeries.from_seasonal(n=mc_n_points.value, period=12, seed=seed),
}

mc_dgp_dropdown = widgets.Dropdown(
    options=list(dgp_options.keys()),
    description='DGP:',
    style={'description_width': 'initial'},
)

mc_n_replications = widgets.IntSlider(
    value=3, min=2, max=50, step=1,
    description='Replications:',
    style={'description_width': 'initial'},
)

mc_base_seed = widgets.IntSlider(
    value=0, min=0, max=1000, step=1,
    description='Base seed:',
    style={'description_width': 'initial'},
)

mc_horizon = widgets.IntSlider(
    value=1, min=1, max=30, step=1,
    description='Horizon (h):',
    style={'description_width': 'initial'},
)

mc_k_first = widgets.BoundedIntText(
    value=10, min=5, max=49_999,
    description='Training window (k_first):',
    style={'description_width': 'initial'},
)

def _on_n_change(change):
    n = change['new']
    mc_k_first.max = n - 1
    if mc_k_first.value >= n:
        mc_k_first.value = max(5, n - 1)

mc_n_points.observe(_on_n_change, names='value')

# Forecaster toggles
mc_use_mean = widgets.Checkbox(value=True, description='Mean (baseline)')
mc_use_arima = widgets.Checkbox(value=True, description='ARIMA')
mc_arima_p = widgets.IntSlider(value=2, min=0, max=10, description='  ARIMA p:')
mc_arima_d = widgets.IntSlider(value=0, min=0, max=2,  description='  ARIMA d:')
mc_arima_q = widgets.IntSlider(value=0, min=0, max=10, description='  ARIMA q:')
mc_use_bayes_ar = widgets.Checkbox(value=True, description='Bayes AR (ridge / MN)')
mc_bayes_ar_p = widgets.IntSlider(value=2, min=1, max=15, description='  BayesAR p:')
mc_bayes_ar_lambda = widgets.FloatLogSlider(
    value=1.0, base=10, min=-3, max=3, step=0.25,
    description='  BayesAR λ:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mode = widgets.Dropdown(
    options=[('Ridge (λI)', 'ridge'), ('Minnesota', 'minnesota')],
    value='ridge',
    description='  BayesAR mode:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mn_decay = widgets.FloatSlider(
    value=2.0, min=0.0, max=4.0, step=0.25,
    description='  MN lag exp:',
    style={'description_width': 'initial'},
)
mc_bayes_ar_mn_rw = widgets.Checkbox(value=True, description='  MN RW mean on φ₁')
mc_use_ssa = widgets.Checkbox(value=True, description='SSA')
mc_use_timesfm = widgets.Checkbox(value=True, description='TimesFM 2.5')

display(
    widgets.HTML('<h4>Monte Carlo settings</h4>'),
    mc_dgp_dropdown, mc_n_replications, mc_base_seed,
    mc_n_points,
    mc_horizon, mc_k_first,
    widgets.HTML('<h4>Forecasters</h4>'),
    mc_use_mean,
    mc_use_arima, mc_arima_p, mc_arima_d, mc_arima_q,
    mc_use_bayes_ar, mc_bayes_ar_p, mc_bayes_ar_lambda,
    mc_bayes_ar_mode, mc_bayes_ar_mn_decay, mc_bayes_ar_mn_rw,
    mc_use_ssa, mc_use_timesfm,
)

HTML(value='<h4>Monte Carlo settings</h4>')

Dropdown(description='DGP:', options=('AR(2)', 'AR(5)', 'ARMA(2,2)', 'ARMA(5,5)', 'Seasonal(12)'), style=Descr…

IntSlider(value=3, description='Replications:', max=50, min=2, style=SliderStyle(description_width='initial'))

IntSlider(value=0, description='Base seed:', max=1000, style=SliderStyle(description_width='initial'))

BoundedIntText(value=100, description='# points (n):', max=50000, min=20, style=DescriptionStyle(description_w…

...

Dropdown(description='  BayesAR mode:', options=(('Ridge (λI)', 'ridge'), ('Minnesota', 'minnesota')), style=D…

FloatSlider(value=2.0, description='  MN lag exp:', max=4.0, step=0.25, style=SliderStyle(description_width='i…

Checkbox(value=True, description='  MN RW mean on φ₁')

Checkbox(value=True, description='SSA')

Checkbox(value=True, description='TimesFM 2.5')

In [5]:
# ── Run Monte Carlo replications ────────────────────────────────

mc_output = widgets.Output()
mc_button = widgets.Button(description='Run Replications', button_style='success')

mc_results: ReplicatedBenchmarkResults | None = None

def _on_mc_run(btn):
    global mc_results
    with mc_output:
        clear_output(wait=True)

        n_pts = mc_n_points.value
        k = mc_k_first.value
        if k >= n_pts:
            print(f'ERROR: Training window (k_first={k}) must be less than # points (n={n_pts}).')
            return

        forecasters = []
        if mc_use_mean.value:
            forecasters.append(MeanForecaster(window=50))
        if mc_use_arima.value:
            forecasters.append(ARIMAForecaster(
                order=(mc_arima_p.value, mc_arima_d.value, mc_arima_q.value)
            ))
        if mc_use_bayes_ar.value:
            forecasters.append(
                BayesianARForecaster(
                    p=mc_bayes_ar_p.value,
                    prior_precision=float(mc_bayes_ar_lambda.value),
                    prior_mode=str(mc_bayes_ar_mode.value),
                    minnesota_lag_decay_exponent=float(mc_bayes_ar_mn_decay.value),
                    minnesota_center_rw=mc_bayes_ar_mn_rw.value,
                )
            )
        if mc_use_ssa.value:
            forecasters.append(SSAForecaster())
        if mc_use_timesfm.value:
            forecasters.append(TimesFMForecaster(max_context=512))

        if not forecasters:
            print('Select at least one forecaster.')
            return

        dgp_factory = dgp_options[mc_dgp_dropdown.value]

        runner = ReplicatedBenchmarkRunner(
            dgp_factory=dgp_factory,
            forecasters=forecasters,
            n_replications=mc_n_replications.value,
            base_seed=mc_base_seed.value,
            k_first=k,
            horizon=mc_horizon.value,
            verbose=True,
        )

        print(f'Running {mc_n_replications.value} replications of '
              f'"{mc_dgp_dropdown.value}" '
              f'(n={n_pts}, k_first={k}, h={mc_horizon.value})...\n')
        mc_results = runner.run()

        print('\n── Aggregate Metrics (mean ± std across seeds) ──')
        display(mc_results.aggregate_metrics())
        print('\n── Pooled Diebold-Mariano (SE loss) ──')
        display(mc_results.pooled_diebold_mariano())

mc_button.on_click(_on_mc_run)
display(mc_button, mc_output)

Button(button_style='success', description='Run Replications', style=ButtonStyle())

Output()

### Monte Carlo Results — Plots

### Monte Carlo — Coverage & Phase 2

Mean empirical coverage across seeds ± one std (when multiple seeds); ``N_valid_mean`` averages evaluation-point counts per seed. **Phase 2** adds ``mean_log_score_kde_mean`` / ``_std`` and sharpness–calibration columns aggregated across seeds.


In [6]:
if mc_results is not None:
    # ── Coverage by nominal level — quantile forecasters only ──────
    cov = mc_results.coverage_table()
    cov = cov[cov["Empirical_coverage_mean"].notna() & (cov["N_valid_mean"] > 0)].reset_index(drop=True)

    if cov.empty:
        print("No quantile forecasts available (enable ARIMA, BayesAR, or TimesFM).")
    else:
        print("── Coverage by nominal level (quantile forecasters only) ──")
        fmt = {
            "Nominal": "{:.0%}",
            "Empirical_coverage_mean": "{:.3f}",
            "Empirical_coverage_std": "{:.3f}",
            "N_valid_mean": "{:.0f}",
        }
        try:
            display(
                cov.style
                .format(fmt, na_rep="—")
                .set_properties(**{"text-align": "right"})
                .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}], overwrite=False)
            )
        except Exception:
            display(cov.round(4))

    # ── Probabilistic scorecard @ 90% — quantile forecasters only ──
    if any(r.quantile_predictions for r in mc_results.per_seed):
        ph2 = mc_results.probabilistic_summary_phase2(nominal=0.9)
        ph2 = ph2[ph2["N_valid_mean"] > 0].reset_index(drop=True)

        ph2 = ph2[[
            "Forecaster",
            "Empirical_coverage_mean",
            "Coverage_error_mean",
            "Mean_PI_width_mean",
            "mean_log_score_kde_mean",
            "mean_log_score_kde_std",
        ]].rename(columns={
            "Empirical_coverage_mean": "Coverage (90%)",
            "Coverage_error_mean":     "Coverage error",
            "Mean_PI_width_mean":      "PI width",
            "mean_log_score_kde_mean": "Log score",
            "mean_log_score_kde_std":  "Log score ±std",
        })

        print("\n── Probabilistic scorecard @ 90% ──")
        num_cols = [c for c in ph2.columns if c != "Forecaster"]
        try:
            display(
                ph2.style
                .format({c: "{:.4f}" for c in num_cols}, na_rep="—")
                .background_gradient(subset=["Log score"], cmap="RdYlGn")
                .set_properties(**{"text-align": "right"})
                .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}], overwrite=False)
            )
        except Exception:
            display(ph2.round(4))
    else:
        print("No quantile forecasts in any replication — enable ARIMA, BayesAR, or TimesFM.")
else:
    print("Run the Monte Carlo replications first (cell above).")

Run the Monte Carlo replications first (cell above).


In [7]:
if mc_results is not None:
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    mc_results.plot_metric_distribution("mse", ax=axes[0])
    mc_results.plot_recursive_mse(ax=axes[1])
    mc_results.plot_pooled_cumulative_error(ax=axes[2])
    plt.tight_layout()
    plt.show()

    _mc_sc = mc_results.replication_scorecard()

    # ── Add RMSE_pooled and RMSE/MEAN ratio ─────────────────────────────────
    _mc_sc["RMSE_pooled"] = np.sqrt(_mc_sc["MSE_pooled"])
    _mean_row = _mc_sc[_mc_sc["Forecaster"] == "Mean"]
    if not _mean_row.empty:
        _mean_rmse = float(_mean_row["RMSE_pooled"].iloc[0])
        _mc_sc["RMSE/MEAN"] = _mc_sc["RMSE_pooled"] / _mean_rmse if _mean_rmse > 0 else float("nan")
    else:
        _mc_sc["RMSE/MEAN"] = float("nan")

    # ── Bar chart: MAE & MSE by model ────────────────────────────────────────
    _forecasters = _mc_sc["Forecaster"].tolist()
    _x = np.arange(len(_forecasters))
    _width = 0.35

    fig2, (ax_bar, ax_ratio) = plt.subplots(1, 2, figsize=(16, 5))

    ax_bar.bar(_x - _width / 2, _mc_sc["MAE_pooled"], _width, label="MAE (pooled)", color="steelblue")
    ax_bar.bar(_x + _width / 2, _mc_sc["MSE_pooled"], _width, label="MSE (pooled)", color="darkorange")
    ax_bar.set_xticks(_x)
    ax_bar.set_xticklabels(_forecasters, rotation=15, ha="right")
    ax_bar.set_title("MAE & MSE by Forecaster (pooled across seeds)")
    ax_bar.set_ylabel("Error")
    ax_bar.legend()
    ax_bar.grid(axis="y", linestyle="--", alpha=0.5)

    # ── Bar chart: RMSE / Mean-baseline RMSE ────────────────────────────────
    _ratio_vals = _mc_sc["RMSE/MEAN"].fillna(1.0).tolist()
    _colors = ["seagreen" if v <= 1.0 else "firebrick" for v in _ratio_vals]
    ax_ratio.bar(_x, _ratio_vals, color=_colors)
    ax_ratio.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="Mean baseline (ratio = 1)")
    ax_ratio.set_xticks(_x)
    ax_ratio.set_xticklabels(_forecasters, rotation=15, ha="right")
    ax_ratio.set_title("RMSE / Mean-baseline RMSE (pooled)  —  green < 1 = beats Mean")
    ax_ratio.set_ylabel("Ratio")
    ax_ratio.legend()
    ax_ratio.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

    # ── Scorecard table ──────────────────────────────────────────────────────
    print("\n── Monte Carlo scorecard (MSE & MAE per seed, pooled, RMSE/MEAN ratio) ──")
    _num = [c for c in _mc_sc.columns if c != "Forecaster"]
    try:
        display(
            _mc_sc.style.format({c: "{:,.5f}" for c in _num}, na_rep="—")
            .set_properties(subset=_num, **{"text-align": "right"})
            .set_table_styles(
                [{"selector": "th", "props": [("text-align", "left")]}],
                overwrite=False,
            )
            .background_gradient(subset=["RMSE/MEAN"], cmap="RdYlGn_r")
        )
    except Exception:
        display(_mc_sc.round(5))
else:
    print('Run the Monte Carlo replications first (cell above).')


Run the Monte Carlo replications first (cell above).
